<center><h1>Soundalgekar_Abhishek_Project_Resnet50</h1></center>

Name: Abhishek Soundalgekar
<br>
Github Username: ABHISHEK SOUNDALGEKAR
<br>
USC ID: 2089011000

Import Packages

In [1]:
pip install -r ../requirements.txt

^C
ERROR: Operation cancelled by user
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import random
import shutil
from pathlib import Path
from typing import Tuple, Dict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import (ResNet50,ResNet101,VGG16,EfficientNetB0)
from tensorflow.keras.layers import (Input,GlobalAveragePooling2D,Dense,Dropout,BatchNormalization,Activation)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, Adam as AdamOpt
from tensorflow.keras.callbacks import (EarlyStopping,ReduceLROnPlateau,ModelCheckpoint
)
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report

import efficientnet.tfkeras as efn
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.regularizers import l2
from tensorflow.keras import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.metrics import AUC


KeyboardInterrupt



# First Model - Resnet50 #

In [ ]:
from pathlib import Path
import shutil, random

def split_into_train_val_test(src_root: Path, dst_root: Path, train_frac=0.8, val_frac_of_train=0.2, seed=123):
    random.seed(seed)
    for cls in sorted((src_root).iterdir()):
        if not cls.is_dir(): continue
        imgs = sorted(cls.iterdir())
        n = len(imgs)
        split_pt = int(train_frac * n)
        train_imgs = imgs[:split_pt]
        test_imgs  = imgs[split_pt:]

        # from train_imgs, take a random val set
        n_val = int(val_frac_of_train * len(train_imgs))
        val_imgs = random.sample(train_imgs, n_val)
        train_imgs = [i for i in train_imgs if i not in val_imgs]

        for subset, files in [("train", train_imgs), ("val", val_imgs), ("test", test_imgs)]:
            for p in files:
                dst = dst_root / subset / cls.name
                dst.mkdir(parents=True, exist_ok=True)
                shutil.copy2(p, dst / p.name)

# Usage:
split_into_train_val_test(
    src_root=Path("../data/RealWaste"),       # your original data folder
    dst_root=Path("../data/WasteSplit")
)

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def preprocess_with_contrast(x):
    # x is a float32 image in [0,255]
    x = tf.image.random_contrast(x, lower=0.8, upper=1.2)
    return tf.keras.applications.resnet50.preprocess_input(x)

augmentor = ImageDataGenerator(
    preprocessing_function=preprocess_with_contrast,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True
)


In [ ]:
DATA_ROOT: Path = Path("../data/WasteSplit")
IMG_DIMS: Tuple[int,int] = (224, 224)
BATCH_SZ: int = 5
NUM_EPOCHS: int = 100

#Augmentation + ResNet50 Preprocessing
augmenter = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input,
    rotation_range=20,            
    width_shift_range=0.2,        
    height_shift_range=0.2,       
    zoom_range=0.2,               
    brightness_range=(0.8, 1.2),  
    horizontal_flip=True
)

def make_generator(
    split: str,
    batch_size: int = BATCH_SZ,
    img_size: Tuple[int,int] = IMG_DIMS,
    shuffle: bool = True
) -> tf.keras.preprocessing.image.DirectoryIterator:
    folder = DATA_ROOT / split
    return augmenter.flow_from_directory(
        directory=str(folder),
        target_size=img_size,
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=shuffle
    )

train_generator = make_generator("train",  batch_size=BATCH_SZ, img_size=IMG_DIMS, shuffle=True)
val_generator   = make_generator("val",    batch_size=BATCH_SZ, img_size=IMG_DIMS, shuffle=False)
test_generator  = make_generator("test",   batch_size=BATCH_SZ, img_size=IMG_DIMS, shuffle=False)

In [ ]:
root_path    = Path("../data/WasteSplit")
train_path   = root_path / "train"
height_px    = 224
width_px     = 224
channels_cnt = 3
input_dims   = (height_px, width_px, channels_cnt)

#Class Count
label_count = sum(1 for d in train_path.iterdir() if d.is_dir())

#Freeze Pretrained Backbone
resnet_core = ResNet50(
    include_top=False,
    weights="imagenet",
    input_shape=input_dims,
    name="resnet50_core"
)
resnet_core.trainable = False

l2_penalty = l2(1e-2)

#Dense → BatchNorm → ReLU Helper
def dense_norm_relu(tensor, units, identifier: str):
    y = Dense(units, kernel_regularizer=l2_penalty, name=f"{identifier}_dense")(tensor)
    y = BatchNormalization(name=f"{identifier}_bn")(y)
    return Activation("relu", name=f"{identifier}_act")(y)

#Classification Head
x = GlobalAveragePooling2D(name="avg_pool")(resnet_core.output)
x = dense_norm_relu(x, 256, "blockA")
x = dense_norm_relu(x, 128, "blockB")
x = Dropout(0.2, name="blockC_dropout")(x)
x = dense_norm_relu(x, 64,  "blockC")

# Final prediction layer
output_layer = Dense(label_count, activation="softmax", name="predictions")(x)

waste_classifier = Model(
    inputs=resnet_core.input,
    outputs=output_layer,
    name="WasteSplit_ResNet50"
)
waste_classifier.summary()

In [ ]:
from tensorflow.keras.optimizers import Adam as OptimizerAdam
from tensorflow.keras.losses     import CategoricalCrossentropy
from tensorflow.keras.metrics    import CategoricalAccuracy

lr_value      = 1e-4
adam_solver   = OptimizerAdam(learning_rate=lr_value)
ce_loss       = CategoricalCrossentropy()
cat_accuracy  = CategoricalAccuracy(name="categorical_accuracy")

classification_engine = waste_classifier
classification_engine.compile(
    optimizer=adam_solver,
    loss=ce_loss,
    metrics=[cat_accuracy]
)

In [ ]:
from tensorflow.keras.metrics import AUC
from tensorflow.keras.callbacks import Callback, EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, BatchNormalization, Dropout, Activation
)
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from pathlib import Path

DATA_DIR = Path("../data/WasteSplit")
H, W     = 224, 224
BATCH    = 5
EPOCHS   = 100
LR       = 1e-4
L2_REG   = 1e-2
PATIENCE = 40

def make_stream(subdir, augment=False):
    args = dict(
        preprocessing_function=tf.keras.applications.resnet.preprocess_input,
        rotation_range=20, width_shift_range=0.2,
        height_shift_range=0.2, zoom_range=0.2,
        brightness_range=(0.8,1.2), horizontal_flip=True
    ) if augment else dict(preprocessing_function=tf.keras.applications.resnet.preprocess_input)
    gen = ImageDataGenerator(**args)
    return gen.flow_from_directory(
        DATA_DIR / subdir,
        target_size=(H, W),
        batch_size=BATCH,
        class_mode="categorical",
        shuffle=(subdir == "train")
    )

train_stream      = make_stream("train", augment=True)
validation_stream = make_stream("val",   augment=True)

# model with AUC
backbone = ResNet50(include_top=False, weights="imagenet", input_shape=(H, W, 3))
backbone.trainable = False

def dense_block(x, units, tag):
    x = Dense(units, kernel_regularizer=l2(L2_REG), name=f"{tag}_dense")(x)
    x = BatchNormalization(name=f"{tag}_bn")(x)
    return Activation("relu", name=f"{tag}_act")(x)

x = GlobalAveragePooling2D()(backbone.output)
x = dense_block(x, 256, "blk1")
x = dense_block(x, 128, "blk2")
x = Dropout(0.2)(x)
x = dense_block(x, 64,  "blk3")
outputs = Dense(train_stream.num_classes, activation="softmax")(x)

model = Model(backbone.input, outputs, name="resnet50_waste")
model.compile(
    optimizer=Adam(LR),
    loss="categorical_crossentropy",
    metrics=["accuracy", AUC(name="auc")]
)

#Callbacks
stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)

class LearningRateLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        logs["learning_rate"] = float(self.model.optimizer.learning_rate.numpy())

lr_logger = LearningRateLogger()

#Train with verbose=1
history = model.fit(
    train_stream,
    validation_data=validation_stream,
    epochs=EPOCHS,
    callbacks=[stop_callback, lr_logger],
    verbose=1
)

In [ ]:
model.save("resnet50_model_final.keras")

In [ ]:
# Convert history to DataFrame 
stats_df = pd.DataFrame(history.history)

#  Extract arrays 
train_losses = stats_df['loss'].to_numpy()
val_losses   = stats_df['val_loss'].to_numpy()
train_accs   = stats_df['accuracy'].to_numpy()
val_accs     = stats_df['val_accuracy'].to_numpy()

# NEW: extract AUC metrics
train_aucs = stats_df['auc'].to_numpy()
val_aucs   = stats_df['val_auc'].to_numpy()

#Epoch index & best epoch 
epochs_list   = np.arange(1, train_losses.size + 1)
best_epoch_ix = int(val_losses.argmin()) + 1

# Loss plot 
plt.figure()
plt.plot(epochs_list, train_losses, label='Training Loss')
plt.plot(epochs_list, val_losses,   label='Validation Loss')
plt.axvline(best_epoch_ix, linestyle='--', label=f'Best Epoch: {best_epoch_ix}')
plt.title('Training vs. Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

#Accuracy plot 
plt.figure()
plt.plot(epochs_list, train_accs, label='Training Accuracy')
plt.plot(epochs_list, val_accs,   label='Validation Accuracy')
plt.axvline(best_epoch_ix, linestyle='--', label=f'Best Epoch: {best_epoch_ix}')
plt.title('Training vs. Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

#AUC plot 
plt.figure()
plt.plot(epochs_list, train_aucs, label='Training AUC')
plt.plot(epochs_list, val_aucs,   label='Validation AUC')
plt.axvline(best_epoch_ix, linestyle='--', label=f'Best Epoch: {best_epoch_ix}')
plt.title('Training vs. Validation AUC')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score
import tensorflow as tf
import pandas as pd
import numpy as np

# Data loader for test set 
test_root = Path("../data/WasteSplit") / "test"
test_only_gen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet50.preprocess_input
).flow_from_directory(
    directory=str(test_root),
    target_size=(H, W),        # Fixed
    batch_size=BATCH,          # Fixed
    class_mode="categorical",
    shuffle=False
)

# Predictions & classification report 
probs_test   = model.predict(test_only_gen, verbose=0)  # Fixed
pred_indices = np.argmax(probs_test, axis=1)
true_indices = test_only_gen.classes

raw_labels    = list(test_only_gen.class_indices.keys())
pretty_labels = [f"{i+1}-{lbl}" for i, lbl in enumerate(raw_labels)]

report_map = classification_report(
    true_indices,
    pred_indices,
    target_names=pretty_labels,
    output_dict=True
)

report_df = pd.DataFrame(report_map).T.round(2)
summaries   = ['accuracy', 'macro avg', 'weighted avg']
class_order = [r for r in report_df.index if r not in summaries]
report_df   = report_df.loc[class_order + summaries]

print("\nResNet50 Test‑Set Classification Report:\n")
print(report_df.to_string())

# Compute & print Macro‑AUC 
y_true_oh = tf.keras.utils.to_categorical(true_indices, num_classes=test_only_gen.num_classes)
macro_auc = roc_auc_score(y_true_oh, probs_test, average='macro', multi_class='ovo')
print(f"\nMacro‑average ROC‑AUC: {macro_auc:.2f}")

In [ ]:
def evaluate_and_report(model, generator, set_name):
    print(f"\nEvaluating {set_name} set...")

    # Predict probabilities
    probs = model.predict(generator, verbose=0)
    preds = np.argmax(probs, axis=1)
    trues = generator.classes

    # Labels
    raw_labels = list(generator.class_indices.keys())
    pretty_labels = [f"{i+1}-{lbl}" for i, lbl in enumerate(raw_labels)]

    # Classification report (per-class)
    report = classification_report(trues, preds, target_names=pretty_labels, output_dict=True)
    report_df = pd.DataFrame(report).T.round(2)

    summaries = ['accuracy', 'macro avg', 'weighted avg']
    class_order = [r for r in report_df.index if r not in summaries]
    report_df = report_df.loc[class_order + summaries]

    print(f"\nResNet50 {set_name} Classification Report:\n")
    print(report_df.to_string())

    # Macro AUC
    y_true_oh = tf.keras.utils.to_categorical(trues, num_classes=generator.num_classes)
    macro_auc = roc_auc_score(y_true_oh, probs, average='macro', multi_class='ovo')

    # Calculate summary metrics
    f1 = f1_score(trues, preds, average='macro')
    precision = precision_score(trues, preds, average='macro')
    recall = recall_score(trues, preds, average='macro')

    print(f"\nResNet50 Model {set_name} Metrics:")
    print(f"ResNet50 Model: {set_name} F1 Score: {f1:.4f}")
    print(f"ResNet50 Model: {set_name} Precision: {precision:.4f}")
    print(f"ResNet50 Model: {set_name} Recall: {recall:.4f}")
    print(f"ResNet50 Model: {set_name} AUC: {macro_auc:.4f}\n")

    return report_df


# Evaluate train set
train_report_df = evaluate_and_report(model, train_loader, "Train")

# Evaluate validation set
val_report_df = evaluate_and_report(model, val_loader, "Validation")

# Evaluate test set
test_report_df = evaluate_and_report(model, test_loader, "Test")

- - -

# Next Model - Resnet101.

In [ ]:
'''
DATA_DIR = Path("../data/WasteSplit")
IMG_DIMS = (224, 224)
BATCH_SZ = 5
EPOCHS   = 1

augmented_preprocessor = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    rotation_range=15,             # small random rotations
    width_shift_range=0.15,        # horizontal shifts
    height_shift_range=0.15,       # vertical shifts
    zoom_range=0.15,               # random zoom
    shear_range=0.1,               # slight shearing
    brightness_range=(0.9, 1.1),   # slight brightness variation
    horizontal_flip=True           # random flips
)

def make_generator(
    subset_name: str,
    shuffle_data: bool = False
) -> tf.keras.preprocessing.image.DirectoryIterator:
    """
    Creates a DirectoryIterator for the given subset ('train', 'val', or 'test').
    """
    return augmented_preprocessor.flow_from_directory(
        directory=str(DATA_DIR / subset_name),
        target_size=IMG_DIMS,
        batch_size=BATCH_SZ,
        class_mode="categorical",
        shuffle=shuffle_data
    )

generators: Dict[str, tf.keras.preprocessing.image.DirectoryIterator] = {
    split: make_generator(split, shuffle_data=(split == "train"))
    for split in ("train", "val", "test")
}

train_loader = generators["train"]
val_loader   = generators["val"]
test_loader  = generators["test"]
'''

from pathlib import Path
from typing import Dict
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.metrics import AUC

DATA_DIR   = Path("../data/WasteSplit")
IMG_DIMS   = (224, 224)
BATCH_SZ   = 5
EPOCHS     = 100

# ─── Augmented preprocessor
augmented_preprocessor = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.resnet.preprocess_input,
    rotation_range=15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.1,
    brightness_range=(0.9, 1.1),
    horizontal_flip=True
)

# ─── Generator factory 
def make_generator(
    subset_name: str,
    shuffle_data: bool = False
) -> tf.keras.preprocessing.image.DirectoryIterator:
    """
    Creates a DirectoryIterator for 'train', 'val', or 'test'.
    """
    return augmented_preprocessor.flow_from_directory(
        directory=str(DATA_DIR / subset_name),
        target_size=IMG_DIMS,
        batch_size=BATCH_SZ,
        class_mode="categorical",
        shuffle=shuffle_data
    )

generators: Dict[str, tf.keras.preprocessing.image.DirectoryIterator] = {
    split: make_generator(split, shuffle_data=(split == "train"))
    for split in ("train", "val", "test")
}

train_loader = generators["train"]
val_loader   = generators["val"]
test_loader  = generators["test"]

# Later, when you compile your model 
from tensorflow.keras.optimizers import Adam

# assume `model` is your built Keras model:
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", AUC(name="auc")]
)

history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS,
    callbacks=[],
    verbose=1
)

In [ ]:
#training History into a DataFrame
resnet101_history_df = pd.DataFrame(history.history)

#1‑based epoch index
epochs = np.arange(1, resnet101_history_df.shape[0] + 1)

#which epoch had the lowest validation loss
best_epoch = int(resnet101_history_df['val_loss'].idxmin()) + 1

#Loss Plot
plt.figure()
plt.plot(epochs, resnet101_history_df['loss'],      label='Training Loss')
plt.plot(epochs, resnet101_history_df['val_loss'],  label='Validation Loss')
plt.axvline(best_epoch, linestyle='--',             label=f'Best Epoch: {best_epoch}')
plt.title('ResNet101 Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

#Accuracy Plot
plt.figure()
plt.plot(epochs, resnet101_history_df['accuracy'],     label='Training Accuracy')
plt.plot(epochs, resnet101_history_df['val_accuracy'], label='Validation Accuracy')
plt.axvline(best_epoch, linestyle='--',                label=f'Best Epoch: {best_epoch}')
plt.title('ResNet101 Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Extract AUC values
train_aucs = resnet101_history_df['auc'].to_numpy()
val_aucs   = resnet101_history_df['val_auc'].to_numpy()

# Plot AUC Curve
plt.figure()
plt.plot(epochs, train_aucs, label='Training AUC')
plt.plot(epochs, val_aucs,   label='Validation AUC')
plt.axvline(best_epoch, linestyle='--', label=f'Best Epoch: {best_epoch}')
plt.title('ResNet101 AUC Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score, f1_score, precision_score, recall_score

def evaluate_and_report(model, generator, set_name):
    print(f"\nEvaluating {set_name} set...")
    probs = model.predict(generator, verbose=0)
    preds = np.argmax(probs, axis=1)
    trues = generator.classes

    raw_labels = list(generator.class_indices.keys())
    pretty_labels = [f"{i+1}-{lbl}" for i, lbl in enumerate(raw_labels)]

    # Classification Report
    report = classification_report(trues, preds, target_names=pretty_labels, output_dict=True)
    report_df = pd.DataFrame(report).T.round(2)

    summary_keys = ['accuracy', 'macro avg', 'weighted avg']
    class_keys = [r for r in report_df.index if r not in summary_keys]
    report_df = report_df.loc[class_keys + summary_keys]

    print(f"\nResNet101 {set_name} Classification Report:\n")
    print(report_df.to_string())

    # AUC, F1, Precision, Recall
    y_true_oh = tf.keras.utils.to_categorical(trues, num_classes=generator.num_classes)
    macro_auc = roc_auc_score(y_true_oh, probs, average='macro', multi_class='ovo')
    f1 = f1_score(trues, preds, average='macro')
    precision = precision_score(trues, preds, average='macro')
    recall = recall_score(trues, preds, average='macro')

    print(f"\nResNet101 Model {set_name} Metrics:")
    print(f"F1 Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"AUC: {macro_auc:.4f}\n")

    return report_df

# ─── Evaluate All Sets
train_report_df = evaluate_and_report(model, train_loader, "Train")
val_report_df   = evaluate_and_report(model, val_loader, "Validation")
test_report_df  = evaluate_and_report(model, test_loader, "Test")


In [ ]:
resnet101_model = classifier_net
resnet101_model.save("Resnet101_model_final.keras")

- - -

# Next Model - VGG16. #

In [ ]:
DATA_DIR: Path        = Path("../data/WasteSplit")
IMAGE_DIMS: Tuple[int,int] = (224, 224)
BATCH_SZ: int         = 5
EPOCHS: int           = 100

#VGG16‑Style Augmentation & Preprocessing
vgg_generator = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    brightness_range=(0.8, 1.2),
    horizontal_flip=True
)

def create_vgg_iterator(
    split: str,
    shuffle: bool = False
) -> tf.keras.preprocessing.image.DirectoryIterator:
    """
    Returns a DirectoryIterator for one of ['train','val','test'] subsets.
    """
    split_path = DATA_DIR / split
    return vgg_generator.flow_from_directory(
        directory=str(split_path),
        target_size=IMAGE_DIMS,
        batch_size=BATCH_SZ,
        class_mode="categorical",
        shuffle=shuffle
    )

vgg_train_iter = create_vgg_iterator("train", shuffle=True)
vgg_val_iter   = create_vgg_iterator("val",   shuffle=False)
vgg_test_iter  = create_vgg_iterator("test",  shuffle=False)

In [ ]:
'''
#Load & freeze the VGG16 backbone
vgg_core = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMAGE_DIMS, 3),
    name="vgg16_core"
)
vgg_core.trainable = False

#Shared L2 penalty and Dense→BN→ReLU helper
l2_penalty = l2(1e-2)
def dense_block(x, units, tag):
    x = Dense(units, kernel_regularizer=l2_penalty, name=f"{tag}_dense")(x)
    x = BatchNormalization(name=f"{tag}_bn")(x)
    return Activation("relu", name=f"{tag}_act")(x)

#classification head
x = GlobalAveragePooling2D(name="vgg_gap")(vgg_core.output)
x = dense_block(x, 256, "vgg_blk1")
x = dense_block(x, 128, "vgg_blk2")
x = Dropout(0.2, name="vgg_dropout")(x)
x = dense_block(x, 64,  "vgg_blk3")

preds = Dense(
    vgg_train_iter.num_classes,
    activation="softmax",
    name="vgg_output"
)(x)

#Assemble & compile the model
vgg_model = Model(inputs=vgg_core.input, outputs=preds, name="WasteSplit_VGG16")
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

#Train with EarlyStopping
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=40,
    restore_best_weights=True,
    verbose=1
)

vgg_history = vgg_model.fit(
    vgg_train_iter,
    validation_data=vgg_val_iter,
    epochs=EPOCHS,
    callbacks=[early_stop],
    verbose=2
)
'''
from tensorflow.keras.metrics import AUC

# ─── Load & freeze the VGG16 backbone 
vgg_core = VGG16(
    include_top=False,
    weights="imagenet",
    input_shape=(*IMAGE_DIMS, 3),
    name="vgg16_core"
)
vgg_core.trainable = False

# ─── Shared L2 penalty and Dense→BN→ReLU helper 
l2_penalty = l2(1e-2)
def dense_block(x, units, tag):
    x = Dense(units, kernel_regularizer=l2_penalty, name=f"{tag}_dense")(x)
    x = BatchNormalization(name=f"{tag}_bn")(x)
    return Activation("relu", name=f"{tag}_act")(x)

# ─── Classification head 
x = GlobalAveragePooling2D(name="vgg_gap")(vgg_core.output)
x = dense_block(x, 256, "vgg_blk1")
x = dense_block(x, 128, "vgg_blk2")
x = Dropout(0.2, name="vgg_dropout")(x)
x = dense_block(x, 64,  "vgg_blk3")

preds = Dense(
    vgg_train_iter.num_classes,
    activation="softmax",
    name="vgg_output"
)(x)

# ─── Assemble & compile the model (with AUC) 
vgg_model = Model(inputs=vgg_core.input, outputs=preds, name="WasteSplit_VGG16")
vgg_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy", AUC(name="auc")]
)

# ─── Train with EarlyStopping 
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=40,
    restore_best_weights=True,
    verbose=1
)

vgg_history = vgg_model.fit(
    vgg_train_iter,
    validation_data=vgg_val_iter,
    epochs=EPOCHS,
    callbacks=[early_stop],
    verbose=2
)

In [ ]:
vgg16_model = classifier_net
vgg16_model.save('vgg16_model_final.keras')

In [ ]:
# VGG16 training history into a DataFrame
vgg_history_df = pd.DataFrame(vgg_history.history)

epoch_indices = np.arange(1, vgg_history_df.shape[0] + 1)

# epoch with the lowest validation loss
optimal_epoch = int(vgg_history_df['val_loss'].idxmin()) + 1

# ─── Loss Curve 
plt.figure()
plt.plot(epoch_indices, vgg_history_df['loss'],      label='Train Loss')
plt.plot(epoch_indices, vgg_history_df['val_loss'],  label='Val Loss')
plt.axvline(optimal_epoch, linestyle='--',           label=f'Best Epoch: {optimal_epoch}')
plt.title('VGG16 Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# ─── Accuracy Curve 
plt.figure()
plt.plot(epoch_indices, vgg_history_df['accuracy'],     label='Train Accuracy')
plt.plot(epoch_indices, vgg_history_df['val_accuracy'], label='Val Accuracy')
plt.axvline(optimal_epoch, linestyle='--',              label=f'Best Epoch: {optimal_epoch}')
plt.title('VGG16 Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# ─── AUC Curve 
train_aucs = vgg_history_df['auc'].to_numpy()
val_aucs   = vgg_history_df['val_auc'].to_numpy()

plt.figure()
plt.plot(epoch_indices, train_aucs, label='Train AUC')
plt.plot(epoch_indices, val_aucs,   label='Val AUC')
plt.axvline(optimal_epoch, linestyle='--',             label=f'Best Epoch: {optimal_epoch}')
plt.title('VGG16 AUC Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.show()

In [ ]:
def evaluate_and_report_vgg(model, generator, set_name):
    print(f"\nEvaluating {set_name} set...")
    probs = model.predict(generator, verbose=0)
    preds = np.argmax(probs, axis=1)
    trues = generator.classes

    raw_labels = list(generator.class_indices.keys())
    pretty_labels = [f"{i+1}-{lbl}" for i, lbl in enumerate(raw_labels)]

    # Classification report with precision, recall, f1-score
    report = classification_report(trues, preds, target_names=pretty_labels, output_dict=True)
    report_df = pd.DataFrame(report).T.round(2)

    summary_labels = ['accuracy', 'macro avg', 'weighted avg']
    class_labels = [r for r in report_df.index if r not in summary_labels]
    report_df = report_df.loc[class_labels + summary_labels]

    print(f"\nVGG16 {set_name} Classification Report:\n")
    print(report_df.to_string())

    # Macro-average ROC AUC
    y_true_oh = tf.keras.utils.to_categorical(trues, num_classes=generator.num_classes)
    macro_auc = roc_auc_score(y_true_oh, probs, average='macro', multi_class='ovo')

    # Summary metrics
    f1 = f1_score(trues, preds, average='macro')
    precision = precision_score(trues, preds, average='macro')
    recall = recall_score(trues, preds, average='macro')

    print(f"\nVGG16 Model {set_name} Metrics:")
    print(f"VGG16 Model: {set_name} F1 Score: {f1:.4f}")
    print(f"VGG16 Model: {set_name} Precision: {precision:.4f}")
    print(f"VGG16 Model: {set_name} Recall: {recall:.4f}")
    print(f"VGG16 Model: {set_name} AUC: {macro_auc:.4f}\n")

    return report_df

train_report_df = evaluate_and_report_vgg(vgg_model, vgg_train_iter, "Train")
val_report_df   = evaluate_and_report_vgg(vgg_model, vgg_val_iter, "Validation")
test_report_df  = evaluate_and_report_vgg(vgg_model, vgg_test_iter, "Test")


- - -

# Next Model - EfficientNetb0.

In [ ]:
DATA_ROOT: Path        = Path("../data/WasteSplit")
IMAGE_SHAPE: Tuple[int, int] = (224, 224)
BATCH_SIZE: int        = 5
EPOCH_COUNT: int       = 100

#EfficientNet‑Style Preprocessor
effnet_preprocessor = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=25,            # random rotations up to ±25°
    width_shift_range=0.2,        # horizontal shifts
    height_shift_range=0.2,       # vertical shifts
    zoom_range=0.15,              # random zooms
    shear_range=0.1,              # slight shearing
    brightness_range=(0.85, 1.15),# brightness jitter
    horizontal_flip=True          # random flips
)

# Loader Factory
def make_data_loader(split: str, shuffle: bool = False):
    """
    Creates an iterator for one of ['train','val','test'].
    """
    folder = DATA_ROOT / split
    return effnet_preprocessor.flow_from_directory(
        directory=str(folder),
        target_size=IMAGE_SHAPE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle
    )
train_loader = make_data_loader("train", shuffle=True)
val_loader   = make_data_loader("val",   shuffle=False)
test_loader  = make_data_loader("test",  shuffle=False)

In [ ]:
pip install -U efficientnet

In [ ]:
ROOT = Path("../data/WasteSplit")
H, W = 224, 224
BATCH, EPOCHS = 5, 100
LR, L2_REG, PATIENCE = 1e-4, 1e-2, 40

# Explicit Generators
def make_gen(subdir, augment=False):
    folder = ROOT / subdir
    if augment:
        gen = ImageDataGenerator(
            preprocessing_function=tf.keras.applications.resnet.preprocess_input,
            rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
            zoom_range=0.2, brightness_range=(0.8,1.2), horizontal_flip=True
        )
    else:
        gen = ImageDataGenerator(
            preprocessing_function=tf.keras.applications.resnet.preprocess_input
        )
    return gen.flow_from_directory(
        directory=str(folder),
        target_size=(H, W),
        batch_size=BATCH,
        class_mode="categorical",
        shuffle=(subdir=="train")
    )

train_loader = make_gen("train", augment=True)
val_loader   = make_gen("val",   augment=True)
test_loader  = make_gen("test",  augment=False)

# ResNet101 + head
backbone = tf.keras.applications.ResNet101(
    weights="imagenet", include_top=False, input_shape=(H,W,3), name="resnet101_base"
)
backbone.trainable = False

def block(x, units, tag):
    x = layers.Dense(units, kernel_regularizer=l2(L2_REG), name=f"{tag}_dense")(x)
    x = layers.BatchNormalization(name=f"{tag}_bn")(x)
    return layers.Activation("relu", name=f"{tag}_act")(x)

x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)
x = block(x, 256, "blk1")
x = block(x, 128, "blk2")
x = layers.Dropout(0.2, name="drop")(x)
x = block(x,  64, "blk3")

output = layers.Dense(
    train_loader.num_classes,
    activation="softmax",
    name="predictions"
)(x)

model = models.Model(backbone.input, output, name="ResNet101_Waste")
model.compile(
    optimizer=optimizers.Adam(LR),
    loss="categorical_crossentropy",
    metrics=["accuracy", AUC(name="auc")]   # added AUC metric
)

# Train with EarlyStopping
es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)
history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS,
    callbacks=[es]
)

In [ ]:

# Convert EfficientNetB0 training history to DataFrame
eff_df = pd.DataFrame(history.history)

# 1‑based epoch index
epoch_nums = np.arange(1, eff_df.shape[0] + 1)

# Epoch with lowest validation loss
best_epoch = int(eff_df['val_loss'].idxmin()) + 1

# ─── Loss Plot 
plt.figure()
plt.plot(epoch_nums, eff_df['loss'],      label='Training Loss')
plt.plot(epoch_nums, eff_df['val_loss'],  label='Validation Loss')
plt.axvline(best_epoch, linestyle='--',   label=f'Best Epoch: {best_epoch}')
plt.title('EfficientNetB0 Loss per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

# ─── Accuracy Plot 
plt.figure()
plt.plot(epoch_nums, eff_df['accuracy'],     label='Training Accuracy')
plt.plot(epoch_nums, eff_df['val_accuracy'], label='Validation Accuracy')
plt.axvline(best_epoch, linestyle='--',      label=f'Best Epoch: {best_epoch}')
plt.title('EfficientNetB0 Accuracy per Epoch')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# ─── AUC Plot 
train_aucs = eff_df['auc'].to_numpy()
val_aucs   = eff_df['val_auc'].to_numpy()

plt.figure()
plt.plot(epoch_nums, train_aucs, label='Training AUC')
plt.plot(epoch_nums, val_aucs,   label='Validation AUC')
plt.axvline(best_epoch, linestyle='--',    label=f'Best Epoch: {best_epoch}')
plt.title('EfficientNetB0 AUC per Epoch')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import roc_auc_score
import tensorflow as tf

def evaluate_and_report(model, generator, set_name, model_name="EfficientNetB0"):
    print(f"\nEvaluating {model_name} on {set_name} set...")

    # Predictions
    probs = model.predict(generator, verbose=0)
    preds = np.argmax(probs, axis=1)
    trues = generator.classes

    # Class label formatting
    raw_labels = list(generator.class_indices.keys())
    readable_labels = [f"{i+1}-{lbl}" for i, lbl in enumerate(raw_labels)]

    # Classification report
    report = classification_report(trues, preds, target_names=readable_labels, output_dict=True)
    report_df = pd.DataFrame(report).T.round(2)

    # Reordering for nicer display
    summary_rows = ['accuracy', 'macro avg', 'weighted avg']
    class_rows   = [r for r in report_df.index if r not in summary_rows]
    report_df    = report_df.loc[class_rows + summary_rows]

    print(f"\n{model_name} {set_name} Classification Report:\n")
    print(report_df.to_string())

    # One-hot encoding for AUC
    y_true_oh = tf.keras.utils.to_categorical(trues, num_classes=generator.num_classes)
    macro_auc = roc_auc_score(y_true_oh, probs, average='macro', multi_class='ovo')

    # Summary metrics
    f1        = f1_score(trues, preds, average='macro')
    precision = precision_score(trues, preds, average='macro')
    recall    = recall_score(trues, preds, average='macro')

    print(f"\n{model_name} {set_name} Set Metrics:")
    print(f"{model_name}: {set_name} F1 Score: {f1:.4f}")
    print(f"{model_name}: {set_name} Precision: {precision:.4f}")
    print(f"{model_name}: {set_name} Recall: {recall:.4f}")
    print(f"{model_name}: {set_name} AUC: {macro_auc:.4f}\n")

    return report_df

# === Usage Example (make sure your generators are defined and use shuffle=False) ===
train_report_df = evaluate_and_report(model, train_loader, "Train", "EfficientNetB0")
val_report_df   = evaluate_and_report(model, val_loader, "Validation", "EfficientNetB0")
test_report_df  = evaluate_and_report(model, test_loader, "Test", "EfficientNetB0")


In [ ]:
EfficientNetB0_model = classifier_net
EfficientNetB0_model.save('EfficientNetB0_model_final.keras')

- - -

In [ ]:
models = {
    "ResNet50":      resnet50_model,
    "ResNet101":     resnet101_model,    # or classifier_net
    "VGG16":         vgg_model,
    "EfficientNetB0": effnet_model
}

loaders = {
    "Train":      train_loader,
    "Validation": val_loader,
    "Test":       test_loader
}

summary = []
for model_name, mdl in models.items():
    for split_name, loader in loaders.items():
        probs = mdl.predict(loader, verbose=0)
        preds = np.argmax(probs, axis=1)
        truths = loader.classes
        
        rep = classification_report(
            truths,
            preds,
            output_dict=True
        )
        macro_prec = rep["macro avg"]["precision"]
        macro_rec  = rep["macro avg"]["recall"]
        macro_f1   = rep["macro avg"]["f1-score"]
        
        y_true_oh = tf.keras.utils.to_categorical(truths, num_classes=loader.num_classes)
        macro_auc = roc_auc_score(y_true_oh, probs, average="macro", multi_class="ovo")
        
        summary.append({
            "Model": model_name,
            "Split": split_name,
            "Precision": round(macro_prec,  2),
            "Recall":    round(macro_rec,   2),
            "F1‑score":  round(macro_f1,    2),
            "AUC":       round(macro_auc,   2)
        })

df_summary = pd.DataFrame(summary)
print(df_summary.pivot(index="Model", columns="Split", values=["Precision","Recall","F1‑score","AUC"]))
